# MUCS Hindi-English ASR: Stage 1B Colab GPU runbook

This notebook prepares a fresh Google Colab GPU runtime, validates the MUCS 2021 Subtask-2 dataset and configured Vakyansh him-4200 checkpoint, runs the existing tests and smoke test, then provides a resumable full-training switch. **It does not claim that Colab, data downloads, model loading, or training have been executed from this repository.**

## Before starting

1. Push the `stage-1b-baseline` branch to GitHub. The clone cell intentionally fails if that exact branch is unavailable; it never falls back to `main` or Stage 1A.
2. In Colab choose **Runtime → Change runtime type → GPU**. A T4 or better is recommended.
3. Allow at least 30 GB free in `/content` for the ~8.8 GB MUCS archives, extracted data, and model cache. Dataset files are kept on ephemeral Colab storage for faster audio reads and must be downloaded again after a runtime reset.
4. Ensure Google Drive has several GB free for checkpoints and run artifacts. This notebook persists checkpoints/logs/configs/reports there; it does not persist the large dataset.

MUCS is CC BY-SA 4.0. Do not copy data, model weights, or credentials into the Git repository.

In [ ]:
import os
import torch
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Select a Colab GPU runtime, reconnect, and rerun this cell.")

device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
capability = torch.cuda.get_device_capability(device)
try:
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        amp_probe = torch.ones((32, 32), device='cuda') @ torch.ones((32, 32), device='cuda')
    torch.cuda.synchronize()
    amp_fp16 = bool(torch.isfinite(amp_probe).all().item())
except (RuntimeError, TypeError) as exc:
    amp_fp16 = False
    print('FP16 autocast probe failed:', repr(exc))
amp_bf16 = torch.cuda.is_bf16_supported()
print({
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": props.name,
    "capability": capability,
    "gpu_total_gib": round(total_bytes / 2**30, 2),
    "gpu_free_gib": round(free_bytes / 2**30, 2),
    "fp16_autocast_available": amp_fp16,
    "bf16_supported": amp_bf16,
})
if free_bytes < 4 * 2**30:
    print("WARNING: less than 4 GiB GPU memory is free. Reduce per-batch samples before training.")
print("/content free GiB:", round(__import__("shutil").disk_usage("/content").free / 2**30, 2))

## Mount Drive, fetch the exact branch, and install dependencies

Colab's preinstalled PyTorch is retained. The requirements file adds the project dependencies and small optional decoder helpers; it deliberately does not install KenLM or replace PyTorch. The repository branch must already be pushed (see the preflight note above).

In [ ]:
from google.colab import drive
import subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/shlok165/hindi-english-code-switching-asr.git'
REPO_BRANCH = 'stage-1b-baseline'
REPO_DIR = Path('/content/hindi-english-code-switching-asr')
remote_heads = subprocess.run(
    ['git', 'ls-remote', '--heads', REPO_URL, REPO_BRANCH],
    check=True, capture_output=True, text=True
).stdout.strip()
if not remote_heads:
    raise RuntimeError(
        f'{REPO_BRANCH!r} is not published at {REPO_URL}. Push the branch, then restart this cell; '
        'the notebook will not silently use a different branch.'
    )
if REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} already exists. Inspect it, then remove/rename only that clone before retrying.')
subprocess.run(
    ['git', 'clone', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    check=True
)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(subprocess.run(['git', 'log', '-1', '--oneline', '--decorate'], check=True, capture_output=True, text=True).stdout)
torch_before = torch.__version__
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'env/requirements-colab.txt'],
    check=True
)
import importlib.metadata as metadata
torch_after = metadata.version('torch')
if torch_after != torch_before.split('+')[0] and torch_after != torch_before:
    raise RuntimeError(f'Installing requirements changed torch from {torch_before} to {torch_after}; aborting GPU setup.')
print('PyTorch retained:', torch_before, '->', torch_after)
print('Repo:', REPO_DIR, '| branch:', REPO_BRANCH, '| commit:', subprocess.run(['git','rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip())

In [ ]:
# Set this before importing project modules: src/__init__.py configures HF_HOME.
DRIVE_ROOT = Path('/content/drive/MyDrive/hindi-english-asr-stage1b')
HF_CACHE = DRIVE_ROOT / 'hf_cache'
CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
RUN_ARTIFACTS = DRIVE_ROOT / 'run_artifacts'
for directory in (HF_CACHE, CHECKPOINTS, RUN_ARTIFACTS, DRIVE_ROOT / 'configs'):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['MUCS_HF_HOME'] = str(HF_CACHE)

# Keep dataset/audio on fast ephemeral local disk; pass Drive as an absolute
# training output path instead of relying on symlinks (not supported by all Drive mounts).
(REPO_DIR / 'data').mkdir(exist_ok=True)

for name in ('data.yaml', 'model_him4200.yaml', 'train_baseline.yaml', 'train_smoke.yaml', 'decode.yaml'):
    source = REPO_DIR / 'configs' / name
    if source.exists():
        (DRIVE_ROOT / 'configs' / name).write_bytes(source.read_bytes())
canary = CHECKPOINTS / '.drive_write_check'
canary.write_text('persistent storage check', encoding='utf-8')
assert canary.read_text(encoding='utf-8') == 'persistent storage check'
canary.unlink()
print('Drive root:', DRIVE_ROOT)
print('Persistent checkpoint path:', CHECKPOINTS)
print('Local dataset root:', repo_data.resolve())

## Download MUCS and build manifests

This fetches train, test, and blind archives (~8.8 GB total), verifies gzip CRCs, records SHA-256 digests, and extracts each official split separately. A 4-hour validation subset is carved from training recordings only. The official test and blind sets are not split, merged, or used for fitting.

In [ ]:
import shutil
free_gib = shutil.disk_usage('/content').free / 2**30
if free_gib < 30:
    raise RuntimeError(f'Only {free_gib:.1f} GiB free in /content; at least 30 GiB is recommended for archive plus extraction.')
subprocess.run([sys.executable, '-m', 'src.data.download', '--only', 'train', 'test', 'blind'], check=True)
subprocess.run([sys.executable, '-m', 'src.data.manifest', '--config', 'configs/data.yaml'], check=True)
subprocess.run([sys.executable, '-m', 'src.models.tokenizer', '--config', 'configs/data.yaml'], check=True)
shutil.copy2(REPO_DIR / 'docs' / 'SHA256SUMS', DRIVE_ROOT / 'SHA256SUMS')
shutil.copy2(REPO_DIR / 'docs' / 'dataset_stats.json', RUN_ARTIFACTS / 'dataset_stats.json')
print('Download, archive CRC/checksum, manifest, and training-only tokenizer steps completed.')

In [ ]:
import json
import soundfile as sf
from src.data.dataset import MucsDataset, read_manifest
from src.data.preprocessing import clean_transcript
from src.utils.config import data_path, load_config

manifest_root = data_path('data/processed/manifests')
splits = {name: read_manifest(manifest_root / f'{name}.jsonl') for name in ('train','val','test','blind')}
split_ids = {name: {row['id'] for row in rows} for name, rows in splits.items()}
minimum_rows = {'train': 40000, 'val': 1800, 'test': 2800, 'blind': 3800}
for name, minimum in minimum_rows.items():
    assert len(splits[name]) >= minimum, f'{name} manifest appears incomplete: {len(splits[name])} rows (< {minimum})'
assert all(len(split_ids[name]) == len(splits[name]) for name in splits), 'duplicate utterance IDs'
for left, right in (('train','val'), ('train','test'), ('train','blind'), ('test','blind'), ('val','test'), ('val','blind')):
    shared = split_ids[left] & split_ids[right]
    assert not shared, f'utterance ID overlap between {left} and {right}: {len(shared)}'
train_recordings = {row['recording'] for row in splits['train']}
val_recordings = {row['recording'] for row in splits['val']}
assert not train_recordings & val_recordings, 'training/validation recording leakage'
assert not ({row['recording'] for row in splits['test']} & {row['recording'] for row in splits['blind']}), 'official test/blind recording overlap'
assert all(row['text'] for name in ('train','val') for row in splits[name]), 'empty training/validation transcript'

missing, bad_rate, unreadable, invalid_segment, non_normalized = [], [], [], [], []
audio_info = {}
for name, rows in splits.items():
    for row in rows:
        audio = data_path(row['audio'])
        if not audio.is_file():
            missing.append((name, row['id'], str(audio)))
            continue
        if audio not in audio_info:
            try:
                info = sf.info(str(audio))
                audio_info[audio] = (info.samplerate, info.frames, info.duration)
            except Exception as exc:
                unreadable.append((str(audio), repr(exc)))
                continue
        rate, frames, duration = audio_info[audio]
        if rate != 16000:
            bad_rate.append((str(audio), rate))
        if row['start'] < 0 or row['end'] <= row['start'] or row['end'] > duration + 0.05:
            invalid_segment.append((name, row['id'], row['start'], row['end'], duration))
        if row['text'] and clean_transcript(row['text']) != row['text']:
            non_normalized.append((name, row['id'], row['text']))
stats_path = REPO_DIR / 'docs' / 'dataset_stats.json'
stats = json.loads(stats_path.read_text(encoding='utf-8'))
audio_audit = stats['audit']
audio_issues = {
    split: {key: audio_audit[split][key] for key in ('recordings_missing_audio','recordings_unreadable','recordings_truncated','recordings_silent','recordings_non_finite') if audio_audit[split][key]}
    for split in audio_audit
}
audio_issues = {split: issues for split, issues in audio_issues.items() if issues}
assert not (missing or bad_rate or unreadable or invalid_segment or non_normalized or audio_issues), {
    'missing': missing[:5], 'bad_sample_rates': bad_rate[:5], 'unreadable': unreadable[:5],
    'invalid_segments': invalid_segment[:5], 'non_normalized_transcripts': non_normalized[:5], 'recording_audit_issues': audio_issues
}

# Sentence repetition is a corpus characteristic; report it, do not move examples across official splits.
train_texts = {row['text'] for row in splits['train'] if row['text']}
test_texts = {row['text'] for row in splits['test'] if row['text']}
blind_texts = {row['text'] for row in splits['blind'] if row['text']}
report = {
    'utterances': {name: len(rows) for name, rows in splits.items()},
    'recordings': {name: len({row['recording'] for row in rows}) for name, rows in splits.items()},
    'empty_transcripts': {name: sum(not row['text'] for row in rows) for name, rows in splits.items()},
    'audio_files_inspected': len(audio_info),
    'sample_rates_hz': sorted({v[0] for v in audio_info.values()}),
    'duration_hours_segments': {name: round(sum(row['duration'] for row in rows)/3600, 3) for name, rows in splits.items()},
    'train_test_sentence_overlap_percent': round(100 * len(train_texts & test_texts) / max(len(test_texts), 1), 2),
    'train_blind_sentence_overlap_percent': round(100 * len(train_texts & blind_texts) / max(len(blind_texts), 1), 2),
    'recording_read_audit': 'PASS (entire recordings read during manifest creation)',
    'integrity': 'PASS',
}
vocab = json.loads((data_path('data/processed/tokenizer') / 'vocab.json').read_text(encoding='utf-8'))
cfg_data = load_config('configs/data.yaml')['data']
filtered = {
    name: MucsDataset(manifest_root / f'{name}.jsonl', vocab, cfg_data['min_seconds'], cfg_data['max_seconds'])
    for name in ('train', 'val')
}
report['trainable_rows_after_CTC_filters'] = {name: len(ds) for name, ds in filtered.items()}
report['CTC_filter_drops'] = {name: ds.dropped for name, ds in filtered.items()}
assert len(filtered['train']) and len(filtered['val']), 'no trainable train or validation rows remain after CTC/duration filters'
print(json.dumps(report, ensure_ascii=False, indent=2))
(RUN_ARTIFACTS / 'colab_dataset_validation.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

## Check model compatibility and batch-memory guidance

The configured checkpoint is `Harveenchadha/vakyansh-wav2vec2-hindi-him-4200`; its identity with the paper's original weights is unverified. Model access/download is verified only when the following cell actually succeeds in Colab. The verifier checks 16 kHz compatibility, the forward output shape, CTC vocabulary retargeting, and the configured blank against real validation audio.

In [ ]:
from src.utils.config import load_config
from src.models.ctc_model import build_model
cfg = load_config('configs/train_baseline.yaml')
for section, key in (('data','extracted_dir'), ('data','manifest_dir'), ('data','tokenizer_dir'), ('training','output_dir')):
    value = cfg[section][key]
    resolved = data_path(value)
    print(f'{section}.{key}: {value} -> {resolved}')
    if section == 'data': assert resolved.exists(), f'Configured data path is missing: {resolved}'
assert cfg['model']['pretrained'] == 'Harveenchadha/vakyansh-wav2vec2-hindi-him-4200'
subprocess.run([sys.executable, '-m', 'src.models.verify_encoder', '--config', 'configs/model_him4200.yaml'], check=True)
encoder_report = REPO_DIR / 'results' / 'encoder_vakyansh-wav2vec2-hindi-him-4200.json'
if encoder_report.is_file(): shutil.copy2(encoder_report, RUN_ARTIFACTS / encoder_report.name)

free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gib = free_bytes / 2**30
if free_gib < 4:
    suggestion = 'Use a smaller GPU or free GPU memory before smoke/training.'
elif total_bytes / 2**30 <= 8:
    suggestion = 'Conservative starting override: --set training.max_samples_per_batch=640000 --set training.gradient_accumulation=3. Monitor OOMs and preserve the override for resume.'
else:
    suggestion = 'Start with configured 1,600,000 samples/batch and accumulation=1; monitor peak memory during smoke/training.'
print('GPU memory guidance (suggestion only; config is unchanged):', suggestion)
print('Configured update size:', cfg['training']['max_samples_per_batch'], 'samples x', cfg['training']['gradient_accumulation'], 'accumulation steps')

## Run tests and the existing full smoke test

The smoke test checks actual audio loading, batching, model forward and finite CTC loss, an optimizer update, checkpoint save/reload, small-batch overfit, validation execution, and training resume equivalence. It can take several minutes and downloads the checkpoint if not cached. A passing result is a pipeline check, **not** a baseline accuracy result.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'src.training.smoke_test'], check=True)
smoke_report = REPO_DIR / 'results' / 'smoke_test.json'
if not smoke_report.is_file():
    raise FileNotFoundError(f'Smoke test reported success but result file is absent: {smoke_report}')
shutil.copy2(smoke_report, RUN_ARTIFACTS / 'smoke_test.json')
result = json.loads(smoke_report.read_text(encoding='utf-8'))
print('Smoke device:', result.get('device'))
print('Smoke steps:', ', '.join(result.get('steps', {}).keys()))
print('Drive copy:', RUN_ARTIFACTS / 'smoke_test.json')

## Full training — explicit opt-in

Training uses the full permitted training manifest and held-out validation manifest; test/blind are not passed to training or checkpoint selection. Checkpoints are written to Google Drive through `data/checkpoints`, with resume checkpoints emitted at each validation interval (every 1,000 updates). If the runtime disconnects after a checkpoint, rerun setup/data/model/smoke cells as needed and run the same command with `--resume auto`.

Set `RUN_FULL_TRAINING=True` only when ready for a multi-hour run. The GPU profile and Stage 1A duration estimates are historical measurements from another machine, not Colab measurements.

In [ ]:
RUN_FULL_TRAINING = False  # Explicitly change to True to start/resume the ~20,000-update run.
TRAIN_COMMAND = [sys.executable, '-m', 'src.training.train', '--config', 'configs/train_baseline.yaml', '--resume', 'auto', '--set', f'training.output_dir={CHECKPOINTS}']
exact_command = ' '.join(TRAIN_COMMAND)
(DRIVE_ROOT / 'configs' / 'training_command.txt').write_text(exact_command + '\n', encoding='utf-8')
print('Exact full-training command:', exact_command)
if RUN_FULL_TRAINING:
    subprocess.run(TRAIN_COMMAND, check=True)
    run_dir = CHECKPOINTS / 'baseline_him4200'
    if not (run_dir / 'best').is_dir():
        raise RuntimeError('Training returned but best checkpoint directory is missing.')
    for filename in ('train_log.csv', 'run_meta.json'):
        path = run_dir / filename
        if path.exists(): print(filename, '->', path)
    shutil.copy2(REPO_DIR / 'configs' / 'train_baseline.yaml', DRIVE_ROOT / 'configs' / 'train_baseline_used.yaml')
    print('Best checkpoint:', run_dir / 'best')
else:
    print('Training not started. Set RUN_FULL_TRAINING=True after checking GPU, disk, model, and smoke-test results.')

## Optional: KenLM setup and decoding

Do not train KenLM from test or blind transcripts. The LM builder accepts only the training manifest. On Colab, KenLM's command-line programs and Python binding may need compilation; this optional cell installs build tools and builds them. Set `BUILD_KENLM=True` only after successful acoustic training (or when explicitly preparing LM development). Tune LM weight, insertion penalty, and beam width on validation only, then freeze settings before test/blind runs.

In [ ]:
BUILD_KENLM = False
if BUILD_KENLM:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake', 'libboost-all-dev', 'zlib1g-dev', 'libbz2-dev', 'liblzma-dev'], check=True)
    kenlm_src = Path('/content/kenlm')
    if not kenlm_src.exists(): subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kpu/kenlm.git', str(kenlm_src)], check=True)
    build_dir = kenlm_src / 'build'
    build_dir.mkdir(exist_ok=True)
    subprocess.run(['cmake', '..', '-DCMAKE_BUILD_TYPE=Release'], cwd=build_dir, check=True)
    subprocess.run(['cmake', '--build', '.', '--target', 'lmplz', 'build_binary', '-j2'], cwd=build_dir, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'pyctcdecode==0.5.0', 'kenlm==0.3.0'], check=True)
    os.environ['PATH'] = str(build_dir / 'bin') + os.pathsep + os.environ['PATH']
    subprocess.run([sys.executable, '-m', 'src.language_model', '--config', 'configs/data.yaml', '--order', '5', '--output-dir', str(DRIVE_ROOT / 'language_model')], check=True)
else:
    print('KenLM build skipped; no LM package/model is claimed to exist.')

## Evaluation and persistent outputs

After successful training, run greedy validation first; with KenLM installed, tune only on validation. Then freeze decoder parameters and evaluate test and blind. Evaluation writes JSONL examples and `<utterance-id> <hypothesis>` files under Drive. Cleaned-text WER/CER from the project utility are diagnostics; official WER/T-WER must use the official scorer and its **raw** references. Blind labels are scoring-only and must never influence training or tuning.

In [ ]:
BEST = CHECKPOINTS / 'baseline_him4200' / 'best'
EVAL_OUT = RUN_ARTIFACTS / 'evaluations'
if BEST.is_dir():
    for split in ('val', 'test', 'blind'):
        subprocess.run([
            sys.executable, '-m', 'src.evaluate', '--split', split,
            '--checkpoint', str(BEST), '--output-dir', str(EVAL_OUT)
        ], check=True)
    print('Greedy evaluation outputs:', EVAL_OUT)
else:
    print('No best checkpoint found; run/resume full training before evaluation.')

print('For KenLM, repeat evaluation on validation only while tuning:')
print(f"python -m src.evaluate --split val --checkpoint '{BEST}' --output-dir '{EVAL_OUT}' --kenlm '{DRIVE_ROOT / 'language_model' / 'train_5gram.klm'}' --beam-width 3000 --lm-weight <VAL_TUNED> --word-insertion <VAL_TUNED>")
print('Freeze those values, then use the same command with --split test and --split blind.')
print('Official blind scorer command (after hypothesis file exists):')
print(f"python -I -X utf8 data/interim/extracted/blind/subtask2_blindtest_wReadme/inference/compute_wer.py --reffile data/interim/extracted/blind/subtask2_blindtest_wReadme/hindi/files/text --hypfile '{EVAL_OUT / 'blind_greedy.hyp'}' --punctmappingfile data/interim/extracted/blind/subtask2_blindtest_wReadme/inference/punct.mapping.txt --wordmappingfile data/interim/extracted/blind/subtask2_blindtest_wReadme/inference/hin2engwords.txt")
print('Test scoring uses the test archive official scorer/reference files; do not substitute cleaned manifest text.')